# FlowOS V2 — GRPO Training on Kaggle

This notebook follows the same high-level pattern as the SF winner notebook:

1. Install dependencies
2. Clone the repo
3. Smoke test the OpenEnv server on HF Space
4. Run GRPO training with `train.py`
5. Plot reward curves from `reward_log.csv`
6. Evaluate base vs trained policy

Environment: [praanjal-flowos-v2.hf.space](https://praanjal-flowos-v2.hf.space)


In [ ]:
!pip install -q -U pip
!pip install -q \
  "trl>=0.11.0" \
  "transformers>=4.45.0" \
  "datasets>=2.21.0" \
  "peft>=0.13.0" \
  "accelerate>=0.34.0" \
  "duckdb>=1.1.0" \
  "openenv-core[core]>=0.2.2" \
  "openai>=1.52.0" \
  "requests>=2.32.0" \
  "python-dotenv>=1.0.1" \
  "matplotlib" \
  "numpy"

In [ ]:
!git clone https://github.com/pranjalparashar/FlowOS_v2.git /kaggle/working/FlowOS_v2
%cd /kaggle/working/FlowOS_v2/developer_control_room
!git pull

In [ ]:
ENV_URL = "https://praanjal-flowos-v2.hf.space"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "outputs/flowos-grpo-kaggle"
DATASET_SIZE = 8
NUM_GENERATIONS = 2
MAX_TURNS = 8

print("ENV_URL:", ENV_URL)
print("MODEL_ID:", MODEL_ID)

In [ ]:
!python -c "from client import DeveloperControlRoomEnv; import asyncio; \
async def main(): \
    env=DeveloperControlRoomEnv(base_url='https://praanjal-flowos-v2.hf.space'); \
    await env.connect(); \
    result=await env.reset(task_id='simulate_csv_report_workflow', scenario_index=0); \
    print(result.observation.scenario_id, result.observation.active_role); \
    await env.close(); \
asyncio.run(main())"

## GRPO Train

This is the first conservative GRPO smoke test. Once this works, scale up to:

- `MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"`
- `DATASET_SIZE = 12`
- `MAX_TURNS = 10`


In [ ]:
!CUDA_VISIBLE_DEVICES=0 python train.py \
  --model-id {MODEL_ID} \
  --env-url {ENV_URL} \
  --task-scope simulate_csv_report_workflow \
  --dataset-size {DATASET_SIZE} \
  --num-generations {NUM_GENERATIONS} \
  --num-epochs 1 \
  --max-turns {MAX_TURNS} \
  --max-new-tokens 96 \
  --per-device-train-batch-size 1 \
  --gradient-accumulation-steps 4 \
  --save-steps 10 \
  --logging-steps 1 \
  --output-dir {OUTPUT_DIR} \
  --report-to none

## Reward Curves

This mirrors the SF winner workflow: train writes `reward_log.csv`, then we plot it.

In [ ]:
!python plot_rewards.py {OUTPUT_DIR}/reward_log.csv --out {OUTPUT_DIR}/reward_curve.png

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.image import imread

img = imread(f"{OUTPUT_DIR}/reward_curve.png")
fig, ax = plt.subplots(figsize=(12, 6))
ax.imshow(img)
ax.axis("off")
plt.show()

## Evaluate Base vs Trained

In [ ]:
!python eval.py \
  --model-id {MODEL_ID} \
  --checkpoint-path /kaggle/working/FlowOS_v2/developer_control_room/{OUTPUT_DIR} \
  --env-url {ENV_URL} \
  --task-scope simulate_csv_report_workflow \
  --episodes 7 \
  --max-turns 12 \
  --results-dir outputs/eval_compare_grpo

In [ ]:
!cat outputs/eval_compare_grpo/eval_summary.json

## Optional: Strict Eval Without Fallback

In [ ]:
!python eval.py \
  --model-id {MODEL_ID} \
  --checkpoint-path /kaggle/working/FlowOS_v2/developer_control_room/{OUTPUT_DIR} \
  --env-url {ENV_URL} \
  --task-scope simulate_csv_report_workflow \
  --episodes 7 \
  --max-turns 12 \
  --disable-fallback \
  --results-dir outputs/eval_compare_grpo_nofallback

In [ ]:
!cat outputs/eval_compare_grpo_nofallback/eval_summary.json